1.导入相关的包


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as functional
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from dataclasses import dataclass
import torch.nn.functional as F
import math

torch.manual_seed(1024)

2.定义GPT的一些参数

In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 512
    batch_size: int = 4  #每次训练喂给模型的数据量，这个值很小，通常用于显存受限的设备或调试。
    n_layer: int = 2
    n_head: int = 12   #注意力机制中的“头”数，每个头关注不同的语义关系。
    n_embd: int = 768   #每个词被转换成的向量维度，影响模型表达能力
    hidden_dim: int = n_embd
    dropout: float = 0.1  #防止过拟合的正则化手段。
    head_size: int = n_embd // n_head  #每个注意力头的维度，由 n_embd / n_head 计算得出（768/12=64）
    vocab_size: int = 50257


3.定义GPT的结构

In [ ]:
#1.single head attention
class  SingleHeadAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.key = nn.Linear(config.hidden_dim,config.head_size)
        self.value = nn.Linear(config.hidden_dim, config.head_size)
        self.query = nn.Linear(config.hidden_dim, config.head_size)
        self.head_size = config.head_size


        self.register_buffer(
            "attention_mask",
            torch.tril(
                torch.ones(config.block_size,config.block_size)
                )
            )
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        batch_size,seq_len,hidden_dim = x.size()
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        weight = q @ k.transpose(-2,-1)
        weight = weight.masked_fill(
            self.attention_mask[:seq_len, :seq_len] == 0,
            float("-inf") 
        )
        weight = F.softmax(weight,dim=-1) / math.sqrt(self.head_size)

        weight = self.dropout(weight)
        output = weight @ v
        return output

#2. multi head attention
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                SingleHeadAttention(config)
                for _ in range(config.n_head)
            ]
        )
        self.proj = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        output = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )
        output = self.proj(output)
        output = self.dropout(output)
        return output
    
 #3. feed forward (MLP)
class FeedForward(nn.Module):
     def __init__(self, config):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(config.hidden_dim, 4 * config.hidden_dim),
                nn.GELU(),
                nn.Linear(4 * config.hidden_dim, config.hidden_dim),
                nn.Dropout(config.dropout)
            )
     def forward(self, x):
            return self.net(x)
        
#4.block
class Block(nn.Module):
    def __init__(self, config):
            super().__init__()
            self.att = MultiHeadAttention(config)
            self.ffn = FeedForward(config)
            self.ln1 = nn.LayerNorm(config.hidden_dim)
            self.ln2 = nn.LayerNorm(config.hidden_dim)

    def forward(self, x):
            x = x + self.att(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
            return x
        
#5.GPT
class GPT(nn.Module):
    def __init__(self, config):
            super().__init__()
            self.token_embedding_table = nn.Embedding(config.vocab_size, config.n_embd)
            self.position_embedding_table = nn.Embedding(config.block_size, config.n_embd)
            self.blocks = nn.Sequential(
                *[Block(config)for _ in range(config.n_layer)]
            )
            self.ln_final = nn.LayerNorm(config.n_embd)
            self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
            self.token_embedding_table.weight = self.lm_head.weight

    def _init_weights(self, module):
            if isinstance(module,nn.Linear):
                #初始化正态分布
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
            batch,seq_len = idx.size()
            token_emb = self.token_embedding_table(idx)
            pos_emb = self.position_embedding_table(
                torch.arange(seq_len, device=idx.device)
            )
            x = token_emb + pos_emb
            x = self.blocks(x)
            x = self.ln_final(x)
            logits = self.lm_head(x)
            if targets is None:
                loss = None
            else:
                batch, seq_len, vocab_size = logits.size()
                logits = logits.view(batch * seq_len, vocab_size)
                targets = targets.view(batch * seq_len)
                loss = F.cross_entropy(logits, targets)
            return logits, loss
        
    def generate(self, idx, max_new_tokens):
            pass #TODO

In [ ]:
# 写一个 dataset，为了 Dataloader 准备
class MyDataset(Dataset):
    def __init__(self, path, block_size=512):
        # 我的数据在 /root/fs/mobvoi_seq_monkey_general_open_corpus.jsonl 中，
        # 读取前 1000 行
        import tiktoken
        self.enc = tiktoken.get_encoding("gpt2")
        self.block_size = block_size

        self.eos_token = self.enc.encode(
            "<|endoftext|>",
            allowed_special={"<|endoftext|>"}
        )[0]

        import json

        self.encoded_data = []

        self.max_lines = 1000
        raw_data = []
        with open(path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i >= self.max_lines:
                    break
                try:
                    text = json.loads(line.strip())['text']
                    raw_data.append(text)
                except json.JSONDecodeError:
                    continue
                except Exception as e:
                    continue
        full_encoded = []
        for text in raw_data:
            encoded_text = self.enc.encode(text)
            full_encoded.extend(encoded_text + [self.eos_token])
        
        # 将长文本分割成训练样本
        for i in range(0, len(full_encoded), self.block_size):
            # 多取一个 Token 作为目标
            chunk = full_encoded[i:i+self.block_size+1]
            # 如果长度不够，用 eos_token 填充
            if len(chunk) < self.block_size + 1:
                chunk = chunk + [self.eos_token] * (self.block_size + 1 - len(chunk))
            self.encoded_data.append(chunk)
    
    def __len__(self):
        return len(self.encoded_data)
    
    def __getitem__(self, idx):
        chunk = self.encoded_data[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

    def encode(self, text):
        """将文本编码为token IDs"""
        return self.enc.encode(text)

    def decode(self, ids):
        """将token IDs解码为文本"""
        return self.enc.decode(ids)

In [ ]:
# 数据的格式
"""
{"text":"担任地点省市的区域运营中心的办理作业。承受总部相关KPI查核。\n1、了解新闻职业或媒体相关运营运营岗位，其间，应聘区域运营中心主任有3年以上当地干流媒体作业经验者优先，应聘事务主管有2年以上当地干流媒体作业经验者优先。\n2、交流才能强，抗压才能强，长于处理复杂情况，了解GR作业优先，能独立完结策划计划优先。具有独立开发客户才能。\n北京、天津、河北、山西、黑龙江、吉林、辽宁、上海、江苏、浙江、安徽、江西、福建、山东、河南、湖北、湖南、广东、海南、重庆、四川、贵州、云南、陕西等。"}
"""


'\n{"text":"担任地点省市的区域运营中心的办理作业。承受总部相关KPI查核。\n1、了解新闻职业或媒体相关运营运营岗位，其间，应聘区域运营中心主任有3年以上当地干流媒体作业经验者优先，应聘事务主管有2年以上当地干流媒体作业经验者优先。\n2、交流才能强，抗压才能强，长于处理复杂情况，了解GR作业优先，能独立完结策划计划优先。具有独立开发客户才能。\n北京、天津、河北、山西、黑龙江、吉林、辽宁、上海、江苏、浙江、安徽、江西、福建、山东、河南、湖北、湖南、广东、海南、重庆、四川、贵州、云南、陕西等。"}\n'

In [ ]:
# train data
train_dataset = MyDataset(r'C:\Users\stellaw\Desktop\vibe coding\mobvoi_seq_monkey_general_open_corpus.jsonl')

# split traindataset to train and val
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [0.9, 0.1])

train_loader = DataLoader(train_dataset, batch_size=12, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=12, shuffle=False)

In [ ]:
model = GPT(GPTConfig())
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# 打印模型一共有多少参数

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e6} M")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
# 设置 cosine 学习率
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)

Total parameters: 53.167872 M


In [ ]:
# 训练循环
def train(model, optimizer, scheduler, train_loader, val_loader, device):
    model.train()
    total_loss = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        # 将数据移到设备上
        x, y = x.to(device), y.to(device)
        
        # 前向传播
        logits, loss = model(x, targets=y)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # 调整学习率
        scheduler.step()
        
        total_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss.item():.4f}')
    return total_loss

def eval(model, val_loader, device):
    # 验证
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, targets=y)
            val_loss += loss.item()
    return val_loss


for epoch in range(2):
    train_loss = train(model, optimizer, scheduler, train_loader, val_loader, device)
    val_loss = eval(model, val_loader, device)
    print(f'Epoch: {epoch}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}')

    # 保存模型
    avg_val_loss = val_loss / len(val_loader)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': avg_val_loss,
    }
    # 保存每个epoch的模型
    torch.save(checkpoint, f'checkpoints/model_epoch_{epoch}.pt')

Epoch: 0, Batch: 0, Loss: 10.9851
Epoch: 0, Batch: 100, Loss: 4.8979


KeyboardInterrupt: 